# FinDER Hybrid Retrieval Experiment

This notebook presents part C of the team assignment: compare BM25, dense retrieval, and hybrid retrieval on the same FinDER ground truth. The official `references` field provides retrieval relevance labels, while `answer` provides the gold answer.

**Main conclusion:** on the held-out test set, weighted RRF hybrid retrieval improves Top-5 Recall and Hit Rate over both individual retrievers.

## Experimental scope

- 5,703 FinDER questions
- 5,830 unique expert reference passages used as the common corpus
- 1,128 development questions for selecting the hybrid weight
- 4,575 held-out test questions for final metrics
- BM25 parameters: `k1=1.2`, `b=0.75`
- Dense model: `sentence-transformers/all-MiniLM-L6-v2`
- Hybrid method: weighted Reciprocal Rank Fusion, candidate depth 100, `RRF k=60`

This is a **reference-passage benchmark**. It is appropriate for the team's controlled comparison but is not directly comparable with the paper's full-10-K RAGAS benchmark.

In [ ]:
from pathlib import Path
import pandas as pd

RESULTS = Path("results")
metrics = pd.read_csv(RESULTS / "retrieval_metrics.csv")
tuning = pd.read_csv(RESULTS / "hybrid_weight_tuning_dev.csv")
answers = pd.read_csv(RESULTS / "answer_correctness.csv")

## Hybrid weight tuning on the development set

In [ ]:
tuning[["dense_weight", "precision_at_k", "recall_at_k", "hit_rate_at_k", "mrr_at_k", "ndcg_at_k"]]

The best development-set Recall@5 is obtained with dense weight 0.25. The final hybrid therefore uses 75% BM25 ranking weight and 25% dense ranking weight.

## Held-out test results at Top 5

In [ ]:
test5 = metrics[(metrics["split"] == "test") & (metrics["k"] == 5)].copy()
test5[["method", "precision_at_k", "recall_at_k", "hit_rate_at_k", "mrr_at_k", "ndcg_at_k", "latency_ms_per_query"]]

| Method | Precision@5 | Recall@5 | Hit Rate@5 | MRR@5 | nDCG@5 | Latency ms/query |
|---|---:|---:|---:|---:|---:|---:|
| BM25 | 0.0605 | 0.2887 | 0.2964 | 0.2166 | 0.2319 | 0.19 |
| Dense MiniLM | 0.0451 | 0.2118 | 0.2195 | 0.1582 | 0.1688 | 3.63 |
| Hybrid RRF | **0.0644** | **0.3056** | **0.3145** | **0.2205** | **0.2386** | 3.87 |

Hybrid improves Recall@5 by 1.69 percentage points and Hit Rate@5 by 1.81 percentage points relative to BM25. BM25 remains slightly better at Top 1, while hybrid becomes stronger from Top 3 onward.

## Results across K

In [ ]:
metrics[metrics["split"] == "test"].pivot(index="method", columns="k", values=["precision_at_k", "recall_at_k", "hit_rate_at_k", "mrr_at_k", "ndcg_at_k"])

## Answer correctness

The answer experiment uses one fixed sample of 50 held-out questions, the same Top-3 context rule, and `google/flan-t5-small` for all three methods. This small model produces very low answer scores and zero exact matches, so the results should be treated as a pipeline check, not the final answer-quality conclusion. The team should rerun this section with its shared final answer model and prompt.

In [ ]:
answers

## Paper benchmark for context only

The paper reports RAGAS Context Recall scores of 11.68 for BM25, 17.83 for GTE, 17.36 for multilingual E5, and 25.95 for E5-Mistral. Those scores use the full 10-K paragraph corpus, a representative 10% query subset, and LLM-based RAGAS scoring. They are not conventional Recall@K and cannot be directly compared with the exact relevance metrics above.

## Reproduce the full experiment

Run this cell from the package directory. Model files are downloaded the first time.

In [ ]:
# Reproduce retrieval, export part-C metrics, and refresh the team table
%run reproduce_c_metrics.py

## What to say in the meeting

We used FinDER's expert references as exact retrieval ground truth and kept the corpus, split, and metrics fixed across methods. We tuned the hybrid weight only on the development set. On 4,575 held-out questions, hybrid retrieval improved Recall@5 from 28.87% with BM25 to 30.56%, and Hit Rate@5 from 29.64% to 31.45%. The improvement is modest but consistent at Top 3, Top 5, and Top 10. The next step is to replace MiniLM with a finance-strong embedding model and rerun answer correctness with the team's shared generation model.